In [ ]:
# FIRST CODE
import cv2
import torch
import torchvision
from jetbot import Robot
from PIL import Image
from IPython.display import display, clear_output
import numpy as np
import time

robot = Robot()

# Load model on GPU
model = torchvision.models.detection.ssdlite320_mobilenet_v3_large(pretrained=True)
model.eval().cuda()

pipeline = "nvarguscamerasrc ! video/x-raw(memory:NVMM), width=320, height=240, framerate=30/1 ! nvvidconv ! video/x-raw, format=BGRx ! videoconvert ! appsink"
cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

# --- Tunable ---
FRAME_W       = 320
FRAME_H       = 240
CENTER_X      = FRAME_W // 2
DEAD_ZONE     = 0.15      # 15% either side = no turn
MAX_TURN      = 0.17      # motor cap
FWD_SPEED     = 0.16
EMA_ALPHA     = 0.25      # lower = smoother, laggier
CONF_THRESH   = 0.45      # lower = more detections
MISS_LIMIT    = 6         # frames before position memory expires
DETECT_EVERY  = 3         # run model every N frames
DISPLAY_EVERY = 15
TURN_PULSE    = 0.07      # seconds per turn burst
# ---------------

smoothed_x  = None
miss_count  = 0
frame_count = 0
last_boxes  = []

print("Started")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # --- Detection (every Nth frame) ---
        if frame_count % DETECT_EVERY == 0:
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            tensor = torch.from_numpy(img_rgb).float().permute(2, 0, 1) / 255.0
            tensor = tensor.unsqueeze(0).cuda()

            with torch.no_grad():
                outputs = model(tensor)[0]

            best_cx   = None
            best_score = 0

            for i in range(len(outputs['labels'])):
                label = outputs['labels'][i].item()
                score = outputs['scores'][i].item()
                if label == 1 and score > CONF_THRESH and score > best_score:
                    box = outputs['boxes'][i].cpu().numpy()
                    x1, y1, x2, y2 = map(int, box)
                    best_cx = (x1 + x2) // 2
                    best_score = score
                    last_boxes = [(x1, y1, x2, y2)]

            if best_cx is not None:
                miss_count = 0
                smoothed_x = (EMA_ALPHA * best_cx + (1 - EMA_ALPHA) * smoothed_x
                              if smoothed_x is not None else float(best_cx))
            else:
                miss_count += 1
                if miss_count >= MISS_LIMIT:
                    smoothed_x = None
                    last_boxes = []

        # --- Control ---
        if smoothed_x is not None:
            error = (smoothed_x - CENTER_X) / FRAME_W  # -0.5 to +0.5

            if error < -DEAD_ZONE:
                spd = min(abs(error) * 0.5, MAX_TURN)
                robot.left(spd)
                time.sleep(TURN_PULSE)
                robot.stop()
                print(f"LEFT  err={error:.2f} spd={spd:.2f}")

            elif error > DEAD_ZONE:
                spd = min(error * 0.5, MAX_TURN)
                robot.right(spd)
                time.sleep(TURN_PULSE)
                robot.stop()
                print(f"RIGHT err={error:.2f} spd={spd:.2f}")

            else:
                robot.forward(FWD_SPEED)
                print("FORWARD")

        else:
            robot.stop()
            print("STOP - no person")

        # --- Display ---
        if frame_count % DISPLAY_EVERY == 0:
            disp = frame.copy()
            for (x1, y1, x2, y2) in last_boxes:
                cv2.rectangle(disp, (x1, y1), (x2, y2), (0, 255, 0), 2)
            if smoothed_x is not None:
                cv2.circle(disp, (int(smoothed_x), FRAME_H // 2), 8, (0, 0, 255), -1)
            clear_output(wait=True)
            display(Image.fromarray(cv2.cvtColor(disp, cv2.COLOR_BGR2RGB)))

except KeyboardInterrupt:
    print("Stopped")

robot.stop()
cap.release()

In [ ]:
# streaming via mjeg

import cv2
import torch
import torchvision
from jetbot import Robot
import numpy as np
import time
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

# ─── Shared frame buffer ───────────────────────────────────────────
output_frame = None
lock = threading.Lock()

# ─── MJPEG Server ─────────────────────────────────────────────────
class StreamHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # silence server logs

    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-type', 'multipart/x-mixed-replace; boundary=frame')
        self.end_headers()
        try:
            while True:
                with lock:
                    if output_frame is None:
                        continue
                    _, jpg = cv2.imencode('.jpg', output_frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
                    frame_bytes = jpg.tobytes()
                self.wfile.write(b'--frame\r\n')
                self.wfile.write(b'Content-Type: image/jpeg\r\n\r\n')
                self.wfile.write(frame_bytes)
                self.wfile.write(b'\r\n')
                time.sleep(0.033)  # ~30fps stream
        except:
            pass

def start_server():
    server = HTTPServer(('0.0.0.0', 8080), StreamHandler)
    server.serve_forever()

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("Stream live → http://<jetson-ip>:8080")

# ─── Robot + Model ────────────────────────────────────────────────
robot = Robot()

model = torchvision.models.detection.ssdlite320_mobilenet_v3_large(pretrained=True)
model.eval().cuda()

pipeline = (
    "nvarguscamerasrc ! "
    "video/x-raw(memory:NVMM), width=320, height=240, framerate=30/1 ! "
    "nvvidconv ! video/x-raw, format=BGRx ! videoconvert ! appsink"
)
cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

# ─── Tunable ──────────────────────────────────────────────────────
FRAME_W       = 320
FRAME_H       = 240
CENTER_X      = FRAME_W // 2
DEAD_ZONE     = 0.15
MAX_TURN      = 0.17
FWD_SPEED     = 0.16
EMA_ALPHA     = 0.25
CONF_THRESH   = 0.45
MISS_LIMIT    = 6
DETECT_EVERY  = 3
TURN_PULSE    = 0.07
# ──────────────────────────────────────────────────────────────────

smoothed_x  = None
miss_count  = 0
frame_count = 0
last_boxes  = []

print("Tracking started")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # ── Detection ──
        if frame_count % DETECT_EVERY == 0:
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            tensor = torch.from_numpy(img_rgb).float().permute(2, 0, 1) / 255.0
            tensor = tensor.unsqueeze(0).cuda()

            with torch.no_grad():
                outputs = model(tensor)[0]

            best_cx    = None
            best_score = 0

            for i in range(len(outputs['labels'])):
                label = outputs['labels'][i].item()
                score = outputs['scores'][i].item()
                if label == 1 and score > CONF_THRESH and score > best_score:
                    box = outputs['boxes'][i].cpu().numpy()
                    x1, y1, x2, y2 = map(int, box)
                    best_cx = (x1 + x2) // 2
                    best_score = score
                    last_boxes = [(x1, y1, x2, y2)]

            if best_cx is not None:
                miss_count = 0
                smoothed_x = (
                    EMA_ALPHA * best_cx + (1 - EMA_ALPHA) * smoothed_x
                    if smoothed_x is not None else float(best_cx)
                )
            else:
                miss_count += 1
                if miss_count >= MISS_LIMIT:
                    smoothed_x = None
                    last_boxes = []

        # ── Control ──
        if smoothed_x is not None:
            error = (smoothed_x - CENTER_X) / FRAME_W

            if error < -DEAD_ZONE:
                spd = min(abs(error) * 0.5, MAX_TURN)
                robot.left(spd)
                time.sleep(TURN_PULSE)
                robot.stop()
                print(f"LEFT  err={error:.2f} spd={spd:.2f}")

            elif error > DEAD_ZONE:
                spd = min(error * 0.5, MAX_TURN)
                robot.right(spd)
                time.sleep(TURN_PULSE)
                robot.stop()
                print(f"RIGHT err={error:.2f} spd={spd:.2f}")

            else:
                robot.forward(FWD_SPEED)
                print("FORWARD")
        else:
            robot.stop()
            print("STOP")

        # ── Write to stream buffer ──
        disp = frame.copy()
        for (x1, y1, x2, y2) in last_boxes:
            cv2.rectangle(disp, (x1, y1), (x2, y2), (0, 255, 0), 2)
        if smoothed_x is not None:
            cv2.circle(disp, (int(smoothed_x), FRAME_H // 2), 8, (0, 0, 255), -1)
            # error bar oveurlay
            cv2.line(disp, (CENTER_X, 0), (CENTER_X, FRAME_H), (255, 255, 0), 1)

        with lock:
            output_frame = disp

except KeyboardInterrupt:
    print("Stopped")

robot.stop()
cap.release()

In [ ]:
# streaming via mjeg

import cv2
import torch
import torchvision
from jetbot import Robot
import numpy as np
import time
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

# ─── Shared frame buffer ───────────────────────────────────────────
output_frame = None
lock = threading.Lock()

# ─── MJPEG Server ─────────────────────────────────────────────────
class StreamHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # silence server logs

    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-type', 'multipart/x-mixed-replace; boundary=frame')
        self.end_headers()
        try:
            while True:
                with lock:
                    if output_frame is None:
                        continue
                    _, jpg = cv2.imencode('.jpg', output_frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
                    frame_bytes = jpg.tobytes()
                self.wfile.write(b'--frame\r\n')
                self.wfile.write(b'Content-Type: image/jpeg\r\n\r\n')
                self.wfile.write(frame_bytes)
                self.wfile.write(b'\r\n')
                time.sleep(0.033)  # ~30fps stream
        except:
            pass

def start_server():
    server = HTTPServer(('0.0.0.0', 8080), StreamHandler)
    server.serve_forever()

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("Stream live → http://<jetson-ip>:8080")

# ─── Robot + Model ────────────────────────────────────────────────
robot = Robot()

model = torchvision.models.detection.ssdlite320_mobilenet_v3_large(pretrained=True)
model.eval().cuda()

pipeline = (
    "nvarguscamerasrc ! "
    "video/x-raw(memory:NVMM), width=320, height=240, framerate=30/1 ! "
    "nvvidconv ! video/x-raw, format=BGRx ! videoconvert ! appsink"
)
cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

# ─── Tunable ──────────────────────────────────────────────────────
FRAME_W       = 320
FRAME_H       = 240
CENTER_X      = FRAME_W // 2
DEAD_ZONE     = 0.15
MAX_TURN      = 0.17
FWD_SPEED     = 0.16
EMA_ALPHA     = 0.25
CONF_THRESH   = 0.45
MISS_LIMIT    = 6
DETECT_EVERY  = 3
TURN_PULSE    = 0.07
# ──────────────────────────────────────────────────────────────────

smoothed_x    = None
miss_count    = 0
frame_count   = 0
last_boxes    = []
multi_person  = False          # ← new flag

print("Tracking started")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # ── Detection ──
        if frame_count % DETECT_EVERY == 0:
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            tensor = torch.from_numpy(img_rgb).float().permute(2, 0, 1) / 255.0
            tensor = tensor.unsqueeze(0).cuda()

            with torch.no_grad():
                outputs = model(tensor)[0]

            # Collect ALL confident person detections first
            person_detections = [
                i for i in range(len(outputs['labels']))
                if outputs['labels'][i].item() == 1
                and outputs['scores'][i].item() > CONF_THRESH
            ]

            multi_person = len(person_detections) >= 2   # ← set flag

            if multi_person:
                # Grab all boxes for display, clear tracking state
                last_boxes = [
                    tuple(map(int, outputs['boxes'][i].cpu().numpy()))
                    for i in person_detections
                ]
                smoothed_x = None
                miss_count = 0
                print(f"MULTI-PERSON ({len(person_detections)} detected) — STOPPED")

            elif len(person_detections) == 1:
                i = person_detections[0]
                box = outputs['boxes'][i].cpu().numpy()
                x1, y1, x2, y2 = map(int, box)
                best_cx = (x1 + x2) // 2
                last_boxes = [(x1, y1, x2, y2)]
                miss_count = 0
                smoothed_x = (
                    EMA_ALPHA * best_cx + (1 - EMA_ALPHA) * smoothed_x
                    if smoothed_x is not None else float(best_cx)
                )

            else:
                multi_person = False
                miss_count += 1
                if miss_count >= MISS_LIMIT:
                    smoothed_x = None
                    last_boxes = []

        # ── Control ──
        if multi_person:
            robot.stop()                                  # ← hard stop

        elif smoothed_x is not None:
            error = (smoothed_x - CENTER_X) / FRAME_W

            if error < -DEAD_ZONE:
                spd = min(abs(error) * 0.5, MAX_TURN)
                robot.left(spd)
                time.sleep(TURN_PULSE)
                robot.stop()
                print(f"LEFT  err={error:.2f} spd={spd:.2f}")

            elif error > DEAD_ZONE:
                spd = min(error * 0.5, MAX_TURN)
                robot.right(spd)
                time.sleep(TURN_PULSE)
                robot.stop()
                print(f"RIGHT err={error:.2f} spd={spd:.2f}")

            else:
                robot.forward(FWD_SPEED)
                print("FORWARD")
        else:
            robot.stop()
            print("STOP")

        # ── Write to stream buffer ──
        disp = frame.copy()

        # Draw all detected boxes (red for multi-person, green for single)
        box_color = (0, 0, 255) if multi_person else (0, 255, 0)
        for (x1, y1, x2, y2) in last_boxes:
            cv2.rectangle(disp, (x1, y1), (x2, y2), box_color, 2)

        # Overlay warning text when multiple people detected
        if multi_person:
            cv2.putText(
                disp, f"MULTI-PERSON: STOPPED",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 0, 255), 2
            )
        elif smoothed_x is not None:
            cv2.circle(disp, (int(smoothed_x), FRAME_H // 2), 8, (0, 0, 255), -1)
            cv2.line(disp, (CENTER_X, 0), (CENTER_X, FRAME_H), (255, 255, 0), 1)

        with lock:
            output_frame = disp

except KeyboardInterrupt:
    print("Stopped")

robot.stop()
cap.release()

Stream live → http://<jetson-ip>:8080
